# 1. Read .snx file as a VCM; read .gfc file as ground truth

In [ ]:
import pathlib

import cartopy
import numpy as np
from matplotlib import pyplot as plt

import sagea
from sagea import SHC

"""define paths"""
date_month = "2008-03"
lmax = 60

path_snx = pathlib.Path(f"/Volumes/ShuhaoWork/GRACE_NEQ/ITSG_SINEX_n96/ITSG-Grace2018_n96_{date_month}.snx")
path_his = pathlib.Path(f"/Volumes/ShuhaoWork/ESM_monthly_mean/HIS_n60/esm_HIS_monthly_{date_month}.gfc")

vcm, _ = sagea.io.read_sinex_cov(path_snx, lmax=lmax)  # this may take a few tens of seconds
shc_truth = SHC.io.from_gfc(path_his, lmax=lmax, key="gfc")


# 2. Generate noise samples from a vcm, and add them on the ground truth

In [ ]:
nsample = 100

shc_with_noise = SHC.generate.normal_from_vcm(vcm, nsample=nsample, mean=shc_truth)

# 3. postprocessing to EWH fields; EWHs at discrete points

In [ ]:
shc_filtered = shc_with_noise.filter.gaussian(300)
shc_filtered_ewh = shc_filtered.convert(from_type="Geopotential", to_type="EWH")

# gridded EWH fields
grid = shc_filtered_ewh.synthesize.to_grid(grid_space=1)

# evaluate EWHs at discrete points
lats = np.array([
    58.5, 57.5, 57.5, 57.5, 56.5, 56.5, 56.5, 56.5, 55.5, 55.5,
])

lons = np.array([
    26.5, 26.5, 27.5, 28.5, 25.5, 26.5, 27.5, 28.5, 27.5, 28.5,
])
ewh_discrete = shc_filtered_ewh.synthesize.evaluate(lats, lons)

# 4. Statistics

In [ ]:
import cartopy

# plot global STD map
lon2d, lat2d = np.meshgrid(grid.lon, grid.lat)

std_grid_value = np.std(grid.value, axis=0)
grid_std = sagea.GRD(std_grid_value * 100, lat=grid.lat, lon=grid.lon)  # EWH in unit [cm]

grid_std.plot(
    vmin=0, vmax=6,
    projection=cartopy.crs.Robinson(),
    cmap="Reds",
)

# plot variance-covariance matrix at discrete points
ewh_discrete *= 100  # EWH in unit [cm]
cov_discrete = np.cov(ewh_discrete.T)
plt.imshow(cov_discrete, cmap="RdBu_r")
plt.colorbar()
plt.show()
